# Métricas de regresión: evaluar predicciones numéricas

<a href="https://colab.research.google.com/" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

En clasificación predecíamos categorías. En regresión, el modelo predice un valor numérico continuo. En este cuaderno estimaremos el **costo de mantenimiento de una pieza industrial** a partir de mediciones simuladas y estudiaremos las métricas principales: **MAE, MSE, RMSE, R² y MAPE**.


## Objetivos del cuaderno

Al finalizar podrás:

- distinguir el valor real, la predicción y el residuo;
- calcular MAE, MSE, RMSE, R² y MAPE;
- interpretar las unidades y el significado operativo de cada métrica;
- entender por qué una sola métrica no basta;
- comparar un modelo lineal con un modelo de bosque aleatorio;
- analizar residuos, valores atípicos y generalización mediante validación cruzada.


## 1. Importar las herramientas

Este cuaderno está preparado para Google Colab. No requiere subir archivos ni instalar paquetes adicionales.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, median_absolute_error,
    mean_absolute_percentage_error, r2_score,
)
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


**Interpretación ampliada:** Se cargan las herramientas de generación, modelado y evaluación. La semilla fija hace reproducibles los resultados.


**Interpretación de la salida:** se cargan las funciones para generar datos, entrenar dos regresores y evaluar errores en las mismas unidades del costo. La semilla fija permite reproducir la división y las métricas.


## 2. Preparar un problema de regresión

Cada fila representa una pieza. Las variables simuladas pueden interpretarse como horas de uso, vibración, temperatura, carga y otras mediciones del proceso. El objetivo es el costo de mantenimiento. Añadiremos una pequeña relación no lineal para que la comparación entre modelos sea interesante.


In [ ]:
X, costo_base = make_regression(
    n_samples=1200, n_features=8, n_informative=6,
    noise=18, bias=120, random_state=RANDOM_STATE,
)

# Componente no lineal para representar un proceso industrial más realista.
costo = costo_base + 12 * np.sin(X[:, 0]) + 8 * (X[:, 1] ** 2)
# Desplazamos el objetivo para que todos los costos sean positivos.
costo = costo - costo.min() + 50

X_train, X_test, y_train, y_test = train_test_split(
    X, costo, test_size=0.30, random_state=RANDOM_STATE,
)
print(f'Piezas totales: {len(costo):,}')
print(f'Costo mínimo: {costo.min():.2f}')
print(f'Costo máximo: {costo.max():.2f}')
print(f'Promedio del costo: {costo.mean():.2f}')
print(f'Tamaño de entrenamiento: {len(y_train):,}')
print(f'Tamaño de prueba: {len(y_test):,}')


**Interpretación ampliada:** El rango, promedio y tamaños de los conjuntos sitúan las métricas en la escala real del costo y confirman la separación entre entrenamiento y prueba.


**Interpretación de la salida:** el objetivo es continuo, no una etiqueta. El rango y el promedio ayudan a poner las métricas en contexto: un error de 10 unidades puede ser pequeño o grande según la escala típica del costo. La separación de prueba reserva piezas que el modelo no verá durante el entrenamiento.


## 3. Explorar la variable objetivo

Antes de entrenar conviene observar la distribución de los costos. Esto ayuda a detectar asimetrías, concentraciones y valores extremos que pueden influir en MAE, MSE y RMSE.


In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ejes[0].hist(y_train, bins=30, color='#4C78A8', alpha=0.8)
ejes[0].set_title('Distribución del costo en entrenamiento')
ejes[0].set_xlabel('Costo de mantenimiento')
ejes[0].set_ylabel('Número de piezas')
ejes[0].grid(alpha=0.2)

ejes[1].boxplot(y_train, vert=False)
ejes[1].set_title('Rango y posibles valores extremos')
ejes[1].set_xlabel('Costo de mantenimiento')
ejes[1].grid(alpha=0.2)
plt.tight_layout(); plt.show()


**Interpretación ampliada:** El histograma y boxplot permiten identificar concentración, dispersión y posibles valores extremos del costo.


**Interpretación de la salida:** el histograma muestra dónde se concentran los costos; el boxplot resume mediana, dispersión y posibles valores extremos. Los valores alejados pueden tener un impacto grande en MSE y RMSE porque ambos elevan al cuadrado los errores.


## 4. Entrenar el modelo lineal

Comenzaremos con una regresión lineal como línea base. El Pipeline estandariza las variables antes de ajustar el modelo, aunque las métricas finales siempre se expresarán en la escala original del costo.


In [ ]:
modelo_lineal = Pipeline([
    ('escalador', StandardScaler()),
    ('regresion', LinearRegression()),
])
modelo_lineal.fit(X_train, y_train)
print('Modelo lineal entrenado.')


**Interpretación ampliada:** La confirmación indica que la regresión lineal quedó ajustada y lista para generar predicciones.


**Interpretación de la salida:** el modelo aprendió una relación promedio entre las mediciones y el costo. La regresión lineal es útil como referencia: un modelo más complejo debe demostrar una mejora en datos no vistos, no solo ajustarse mejor al entrenamiento.


## 5. Obtener predicciones y residuos

Una regresión produce un número para cada pieza. El residuo será la diferencia entre el costo real y el costo predicho:


<div style="background-color:#e8f4f8;border-left:6px solid #31708f;padding:12px;font-size:1.1em"><b>Definición clave</b><br><br>$$Residuo_i = y_i - \hat{y}_i$$<br><br>Un residuo positivo significa que el modelo subestimó el costo; uno negativo significa que lo sobreestimó.</div>


In [ ]:
pred_lineal = modelo_lineal.predict(X_test)
residuos_lineal = y_test - pred_lineal

tabla_predicciones = pd.DataFrame({
    'costo_real': y_test,
    'costo_predicho': pred_lineal,
    'residuo': residuos_lineal,
    'error_absoluto': np.abs(residuos_lineal),
})
tabla_predicciones.head(10).round(2)


**Interpretación ampliada:** Cada fila contrasta costo real, predicción, residuo y error absoluto; el signo indica sobreestimación o subestimación.


**Interpretación de la salida:** cada fila compara una pieza real con la estimación del modelo. El error absoluto ignora si se sobreestimó o subestimó y mide únicamente la distancia. Para revisar un modelo, conviene observar tanto el tamaño del error como su signo.


## 6. Fórmulas principales de regresión

Las métricas responden preguntas diferentes y están en escalas distintas:


<div style="background-color:#fff3cd;border-left:6px solid #f0ad4e;padding:12px;font-size:1.05em"><b>MAE — Error absoluto medio</b><br>$$MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i-\hat{y}_i|$$<br>Promedio de cuánto nos equivocamos, en unidades del costo.<br><br><b>MSE — Error cuadrático medio</b><br>$$MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2$$<br>Penaliza fuertemente los errores grandes y queda en unidades cuadradas.<br><br><b>RMSE — Raíz del error cuadrático medio</b><br>$$RMSE = \sqrt{MSE}$$<br>Penaliza errores grandes, pero vuelve a las unidades originales del costo.</div>


<div style="background-color:#dff0d8;border-left:6px solid #3c763d;padding:12px;font-size:1.05em"><b>R² — Coeficiente de determinación</b><br>$$R^2 = 1 - \frac{\sum(y_i-\hat{y}_i)^2}{\sum(y_i-\bar{y})^2}$$<br>Compara el modelo contra una referencia que siempre predice el promedio.<br><br><b>MAPE — Error porcentual absoluto medio</b><br>$$MAPE = \frac{100}{n}\sum \left|\frac{y_i-\hat{y}_i}{y_i}\right|$$<br>Expresa el error relativo en porcentaje; debe usarse con cuidado cuando los valores reales son cero o muy cercanos a cero.</div>


## 7. Calcular las métricas manualmente

Aplicaremos las fórmulas directamente sobre los residuos para hacer visible qué está calculando cada métrica.


In [ ]:
error_absoluto = np.abs(residuos_lineal)
error_cuadratico = residuos_lineal ** 2

mae_manual = error_absoluto.mean()
mse_manual = error_cuadratico.mean()
rmse_manual = np.sqrt(mse_manual)
r2_manual = 1 - (error_cuadratico.sum() / ((y_test - y_test.mean()) ** 2).sum())
mape_manual = np.mean(error_absoluto / np.abs(y_test)) * 100

metricas_manuales = pd.DataFrame({
    'métrica': ['MAE', 'MSE', 'RMSE', 'R2 / RSQ', 'MAPE (%)'],
    'valor': [mae_manual, mse_manual, rmse_manual, r2_manual, mape_manual],
})
metricas_manuales.round(4)


**Interpretación ampliada:** Las fórmulas muestran que MAE es interpretable en unidades, RMSE enfatiza errores grandes, R² compara contra el promedio y MAPE expresa error relativo.


**Interpretación de la salida:** MAE es el error promedio en unidades de costo; RMSE está en las mismas unidades, pero aumenta más cuando existen errores grandes; MSE queda al cuadrado y es menos intuitivo para comunicar. R² indica cuánto mejora el modelo frente a predecir siempre el promedio; MAPE representa el error relativo promedio.


## 8. Comprobar las métricas con scikit-learn

La implementación de la biblioteca debe coincidir con el cálculo manual. Esta comprobación reduce errores de fórmula y deja listo un patrón reutilizable para otros modelos.


In [ ]:
metricas_sklearn = pd.DataFrame({
    'métrica': ['MAE', 'MSE', 'RMSE', 'R2', 'MAPE'],
    'manual': [
        mae_manual, mse_manual, rmse_manual, r2_manual, mape_manual,
    ],
    'scikit_learn': [
        mean_absolute_error(y_test, pred_lineal),
        mean_squared_error(y_test, pred_lineal),
        mean_squared_error(y_test, pred_lineal) ** 0.5,
        r2_score(y_test, pred_lineal),
        mean_absolute_percentage_error(y_test, pred_lineal) * 100,
    ],
})
metricas_sklearn.round(5)


**Interpretación ampliada:** La tabla manual resume el error del modelo lineal en cinco perspectivas complementarias.


**Interpretación de la salida:** las columnas manual y scikit-learn deben ser iguales salvo diferencias mínimas de redondeo. Si no coincidieran, habría que revisar la fórmula, el signo del residuo o la escala utilizada antes de interpretar el modelo.


## 9. Comparar valores reales contra predicciones

La gráfica ideal tendría todos los puntos sobre la diagonal. La distancia vertical entre un punto y la diagonal representa el error de esa pieza.


In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, pred_lineal, alpha=0.65, color='#4C78A8', label='Piezas de prueba')
limites = [min(y_test.min(), pred_lineal.min()), max(y_test.max(), pred_lineal.max())]
plt.plot(limites, limites, '--', color='black', label='Predicción perfecta')
plt.title('Valores reales frente a predicciones')
plt.xlabel('Costo real')
plt.ylabel('Costo predicho')
plt.legend(); plt.grid(alpha=0.25); plt.show()


**Interpretación ampliada:** Los resultados manuales y de scikit-learn deben coincidir salvo redondeo; esto valida las fórmulas.


**Interpretación de la salida:** puntos cercanos a la diagonal representan predicciones precisas. Una nube muy dispersa indica errores grandes. Si los puntos se curvan o se separan sistemáticamente en un extremo, el modelo lineal puede estar capturando mal una relación no lineal o una variación de escala.


## 10. Analizar los residuos

Los residuos ayudan a diagnosticar el modelo. Idealmente deberían estar centrados alrededor de cero, sin una tendencia clara respecto a la predicción.


In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ejes[0].scatter(pred_lineal, residuos_lineal, alpha=0.65, color='#E45756')
ejes[0].axhline(0, color='black', linestyle='--')
ejes[0].set_title('Residuos frente a predicción')
ejes[0].set_xlabel('Costo predicho'); ejes[0].set_ylabel('Residuo')
ejes[0].grid(alpha=0.25)

ejes[1].hist(residuos_lineal, bins=30, color='#72B7B2', alpha=0.85)
ejes[1].axvline(0, color='black', linestyle='--')
ejes[1].set_title('Distribución de residuos')
ejes[1].set_xlabel('Residuo'); ejes[1].set_ylabel('Número de piezas')
ejes[1].grid(alpha=0.25)
plt.tight_layout(); plt.show()


**Interpretación ampliada:** La nube ideal se acerca a la diagonal; su dispersión y curvatura revelan errores, sesgo o relaciones no lineales.


**Interpretación de la salida:** un patrón de embudo sugiere que la variabilidad del error cambia según el nivel del costo; una curva sugiere que falta capturar no linealidad; una distribución muy desplazada de cero indica sesgo sistemático. Los residuos no solo sirven para producir métricas: ayudan a explicar por qué el modelo falla.


## 11. Comparar un modelo lineal con un bosque aleatorio

El bosque puede capturar relaciones no lineales e interacciones. Lo compararemos con las mismas métricas y sobre el mismo conjunto de prueba.


In [ ]:
modelo_bosque = RandomForestRegressor(
    n_estimators=250, max_depth=12, min_samples_leaf=3,
    random_state=RANDOM_STATE, n_jobs=-1,
)
modelo_bosque.fit(X_train, y_train)
pred_bosque = modelo_bosque.predict(X_test)

def resumen_regresion(nombre, reales, predichos):
    return {
        'modelo': nombre,
        'MAE': mean_absolute_error(reales, predichos),
        'RMSE': mean_squared_error(reales, predichos) ** 0.5,
        'R2': r2_score(reales, predichos),
        'MAPE (%)': mean_absolute_percentage_error(reales, predichos) * 100,
    }

comparacion = pd.DataFrame([
    resumen_regresion('Regresión lineal', y_test, pred_lineal),
    resumen_regresion('Bosque aleatorio', y_test, pred_bosque),
])
comparacion.round(4)


**Interpretación ampliada:** Los residuos deberían fluctuar alrededor de cero sin patrones; tendencias o embudos señalan problemas de especificación.


**Interpretación de la salida:** para MAE, RMSE y MAPE, valores menores son mejores; para R², valores mayores suelen ser mejores. Si el bosque mejora en prueba, su flexibilidad está capturando parte de la no linealidad. La elección no debe basarse solo en una métrica: también importan interpretabilidad, costo y estabilidad.


## 12. Ver cómo cada métrica ordena los modelos

Las métricas no siempre reaccionan igual. MAE representa el error típico; RMSE da más peso a errores extremos; R² compara contra la variabilidad total del objetivo; MAPE normaliza por el tamaño de cada valor real.


In [ ]:
comparacion_larga = comparacion.set_index('modelo').T
comparacion_larga.plot(kind='bar', subplots=True, layout=(2, 2), figsize=(12, 7), legend=False, sharex=False)
plt.suptitle('Cada métrica ofrece una perspectiva distinta', y=1.02)
plt.tight_layout(); plt.show()


**Interpretación ampliada:** Para MAE, RMSE y MAPE menor es mejor; para R² mayor suele ser mejor. La comparación debe hacerse en prueba.


**Interpretación de la salida:** no compares directamente la altura de MAE con la de R²: tienen escalas y significados diferentes. Usa cada panel según su objetivo. Si RMSE es mucho mayor que MAE, hay errores grandes que merecen investigación.


## 13. Efecto de un valor atípico

Un único error extremo puede afectar mucho a las métricas cuadráticas. Simularemos una pieza cuyo costo real fue excepcionalmente alto, manteniendo las predicciones iguales, para aislar el efecto de ese caso.


In [ ]:
y_test_atipico = y_test.copy()
y_test_atipico[0] = y_test_atipico[0] * 2.5

comparacion_atipico = pd.DataFrame({
    'situacion': ['Prueba original', 'Con un valor atípico'],
    'MAE': [
        mean_absolute_error(y_test, pred_lineal),
        mean_absolute_error(y_test_atipico, pred_lineal),
    ],
    'RMSE': [
        mean_squared_error(y_test, pred_lineal) ** 0.5,
        mean_squared_error(y_test_atipico, pred_lineal) ** 0.5,
    ],
    'R2': [
        r2_score(y_test, pred_lineal),
        r2_score(y_test_atipico, pred_lineal),
    ],
    'MedAE': [
        median_absolute_error(y_test, pred_lineal),
        median_absolute_error(y_test_atipico, pred_lineal),
    ],
})
comparacion_atipico.round(4)


**Interpretación ampliada:** Las gráficas separadas evitan comparar magnitudes incompatibles y permiten leer cada métrica con su propia escala.


**Interpretación de la salida:** RMSE y MSE son sensibles al valor atípico porque elevan el error al cuadrado. MAE cambia de forma más proporcional y MedAE puede permanecer relativamente estable porque usa la mediana de los errores absolutos. Esta comparación ayuda a elegir una métrica coherente con la presencia o ausencia de casos extremos.


## 14. Validación cruzada

Una sola división puede producir una estimación afortunada o desfavorable. La validación cruzada repite la evaluación en varias particiones y permite observar la variabilidad del desempeño.


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    'MAE': 'neg_mean_absolute_error',
    'RMSE': 'neg_root_mean_squared_error',
    'R2': 'r2',
}
resultados_cv = cross_validate(
    modelo_bosque, X, costo, cv=cv, scoring=scoring, n_jobs=-1,
)

resumen_cv = pd.DataFrame({
    'métrica': ['MAE', 'RMSE', 'R2'],
    'promedio': [
        -resultados_cv['test_MAE'].mean(),
        -resultados_cv['test_RMSE'].mean(),
        resultados_cv['test_R2'].mean(),
    ],
    'desviación estándar': [
        resultados_cv['test_MAE'].std(),
        resultados_cv['test_RMSE'].std(),
        resultados_cv['test_R2'].std(),
    ],
})
resumen_cv.round(4)


**Interpretación ampliada:** El valor atípico muestra por qué RMSE/R² pueden cambiar más que MAE o MedAE.


**Interpretación de la salida:** el promedio resume el desempeño esperado a través de cinco particiones y la desviación estándar muestra su variabilidad. Un modelo con promedio bueno pero desviación muy alta puede ser inestable; la confiabilidad del resultado importa tanto como su valor medio.


## 15. ¿Qué métrica conviene usar?

| Necesidad | Métrica útil | Razón |
|---|---|---|
| Comunicar el error típico en unidades del costo | MAE | Fácil de interpretar y menos sensible a extremos |
| Penalizar mucho errores grandes | RMSE | Amplifica desviaciones grandes |
| Comparar contra una predicción del promedio | R² | Resume mejora relativa frente a una referencia |
| Comunicar error relativo | MAPE | Expresa el error como porcentaje; evitar con ceros |
| Robustez ante valores extremos | MedAE | Usa la mediana de errores absolutos |


## 16. Una pequeña exploración

Modifica el número de árboles, la profundidad o el número de variables y vuelve a ejecutar las celdas de entrenamiento y evaluación. Observa si la mejora en entrenamiento también aparece en prueba y validación cruzada.


In [ ]:
configuraciones = [
    ('Bosque pequeño', RandomForestRegressor(n_estimators=100, max_depth=5, random_state=RANDOM_STATE, n_jobs=-1)),
    ('Bosque mediano', RandomForestRegressor(n_estimators=250, max_depth=12, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1)),
    ('Bosque profundo', RandomForestRegressor(n_estimators=250, max_depth=None, min_samples_leaf=1, random_state=RANDOM_STATE, n_jobs=-1)),
]
resultados_exploracion = []
for nombre, modelo in configuraciones:
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    resultados_exploracion.append(resumen_regresion(nombre, y_test, pred))
pd.DataFrame(resultados_exploracion).round(4)


**Interpretación ampliada:** El promedio de validación cruzada resume desempeño y la desviación estándar muestra estabilidad entre particiones.


**Interpretación de la salida:** compara el cambio en MAE, RMSE, R² y MAPE al modificar la complejidad. Un modelo profundo puede mejorar el ajuste, pero si empeora o se vuelve inestable en prueba puede estar sobreajustando. La configuración adecuada es la que generaliza bien y responde a las necesidades del problema.


## Cierre

MAE comunica el error absoluto típico; MSE y RMSE penalizan especialmente los errores grandes; R² compara el modelo con la predicción del promedio; MAPE expresa el error relativo con precauciones cuando hay ceros; y MedAE ofrece una alternativa robusta.

La evaluación de regresión debe combinar métricas, gráficos de predicciones, análisis de residuos y validación con datos no utilizados durante el entrenamiento.


## Para pensar

1. ¿Por qué RMSE puede ser mucho mayor que MAE?
2. ¿Qué significa un R² negativo en un conjunto de prueba?
3. ¿Cuándo preferirías MAE frente a RMSE?
4. ¿Por qué MAPE puede ser problemático cuando el costo real es cero o muy pequeño?
5. ¿Qué evidencia necesitarías para afirmar que el bosque es mejor que la regresión lineal?
